# Dataset Download

This notebook downloads and organizes the fall detection datasets.

## Datasets:
1. **UR Fall Detection Dataset** - Universidad de Rzeszów, Poland
   - 70 sequences (30 falls, 40 ADL - Activities of Daily Living)
   - Resolution: 640×480
   - FPS: 30
   - Source: http://fenix.ur.edu.pl/~mkepski/ds/uf.html

2. **Le2i Fall Detection Dataset** - Université de Bourgogne, France
   - ~200 videos in varied scenarios (home, office, conference room)
   - Multiple environments and angles
   - Source: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html

## Goals:
1. Download UR Fall Detection dataset
2. Download Le2i Fall Detection dataset
3. Organize files into proper directory structure
4. Verify downloads and create metadata
5. Save to Google Drive (if running in Colab)

## 0. Setup Environment

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")

In [ ]:
# Standard imports
import os
import sys
import requests
import zipfile
import tarfile
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import shutil
from bs4 import BeautifulSoup
from typing import List, Dict
import time

## 1. Configuration

In [ ]:
# Configuration: Google Drive folder name
# Change this if you want to use a different folder name in Google Drive
GOOGLE_DRIVE_PROJECT_FOLDER = 'safeguard-vision-ai'

print(f"✓ Project folder configured: {GOOGLE_DRIVE_PROJECT_FOLDER}")

## 2. Setup Google Drive Integration (For Colab)

In [ ]:
if IN_COLAB:
    print("=" * 60)
    print("Google Colab Environment Detected")
    print("=" * 60)
    
    # Try to detect if running in VS Code
    import subprocess
    try:
        # Check if we can mount Drive (may fail in VS Code)
        from google.colab import drive
        
        # Check if already mounted
        if os.path.exists('/content/drive/MyDrive'):
            print("✓ Google Drive already mounted")
            USE_DRIVE = True
        else:
            print("\n⚠ IMPORTANT: Google Drive mounting")
            print("If you're using VS Code, Drive mounting may not work.")
            print("In that case, you can:")
            print("  1. Download files directly to /content/ (temporary)")
            print("  2. Manually copy files to Drive after download")
            print("  3. Use Google Colab in browser instead")
            print("\nAttempting to mount Google Drive...")
            
            try:
                drive.mount('/content/drive', force_remount=False)
                print("✓ Google Drive mounted successfully")
                USE_DRIVE = True
            except Exception as e:
                print(f"\n✗ Drive mount failed: {e}")
                print("\n" + "=" * 60)
                print("WORKAROUND FOR VS CODE")
                print("=" * 60)
                print("Since Drive mounting failed, files will be downloaded to:")
                print("  /content/safeguard-vision-ai/data/raw/")
                print("\nAfter download, you can manually copy to Drive:")
                print("  1. Open this notebook in browser: https://colab.research.google.com")
                print("  2. Or download files from Colab and upload to Drive")
                print("=" * 60)
                USE_DRIVE = False
    except Exception as e:
        print(f"✗ Error with Drive: {e}")
        USE_DRIVE = False
    
    # Clone repository
    if not os.path.exists('/content/safeguard-vision-ai'):
        print("\nCloning repository...")
        !git clone https://github.com/hugoangeles0810/safeguard-vision-ai.git
        os.chdir('/content/safeguard-vision-ai')
        
        # Install dependencies
        print("\nInstalling dependencies...")
        !pip install -q -r requirements.txt
        print("✓ Dependencies installed")
    else:
        print("\n✓ Repository already exists")
        os.chdir('/content/safeguard-vision-ai')
    
    print(f"\nCurrent directory: {os.getcwd()}")
    
    # Setup Python path
    repo_src_path = '/content/safeguard-vision-ai/src'
    if repo_src_path not in sys.path:
        sys.path.insert(0, repo_src_path)
    
    # Setup paths based on whether Drive is available
    if USE_DRIVE:
        # Import drive utilities
        try:
            from utils.drive_utils import setup_paths
            print("✓ Drive utilities imported")
            paths = setup_paths(project_name=GOOGLE_DRIVE_PROJECT_FOLDER)
            
            # Use temp dir for downloads, then move to Drive
            download_dir = Path('/content/dataset_downloads')
            download_dir.mkdir(exist_ok=True)
            
            print("\n" + "=" * 60)
            print("Using Google Drive Storage")
            print("=" * 60)
            print(f"✓ Project folder: {GOOGLE_DRIVE_PROJECT_FOLDER}")
            print(f"✓ Temp downloads: {download_dir}")
            print(f"✓ Final destination: {paths['data_raw']}")
            print(f"✓ Drive path: /content/drive/MyDrive/{GOOGLE_DRIVE_PROJECT_FOLDER}/")
        except ImportError as e:
            print(f"Import error: {e}, falling back to local storage")
            USE_DRIVE = False
    
    if not USE_DRIVE:
        # Fallback: use local Colab storage (temporary, lost on disconnect)
        print("\n" + "=" * 60)
        print("Using Local Colab Storage (TEMPORARY)")
        print("=" * 60)
        print("⚠ Warning: Files will be LOST when runtime disconnects!")
        print("⚠ Recommendation: Use browser Colab for Drive integration")
        
        local_data_dir = Path('/content/safeguard-vision-ai/data/raw')
        local_data_dir.mkdir(parents=True, exist_ok=True)
        
        paths = {
            'project_root': Path('/content/safeguard-vision-ai'),
            'data_raw': local_data_dir,
        }
        download_dir = local_data_dir
        
        print(f"\n✓ Data will be saved to: {paths['data_raw']}")
        print("✓ Files saved here are accessible within this session")
        print("\nTo preserve data:")
        print("  1. Download files before session ends")
        print("  2. Or run notebook in browser Colab with Drive mounted")

else:
    # Local environment (not Colab)
    print("=" * 60)
    print("Local Environment Detected")
    print("=" * 60)
    
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    paths = {
        'project_root': project_root,
        'data_raw': project_root / 'data' / 'raw',
    }
    download_dir = project_root / 'data' / 'downloads'
    download_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"✓ Download location: {download_dir}")
    print(f"✓ Destination: {paths['data_raw']}")

print(f"\n{'='*60}")
print(f"Final Configuration:")
print(f"{'='*60}")
print(f"Data destination: {paths['data_raw']}")
print(f"Download directory: {download_dir}")
print(f"{'='*60}")

## 3. Helper Functions

In [ ]:
def download_file(url: str, destination: Path, description: str = "Downloading") -> Path:
    """
    Download a file from a URL with progress bar.
    
    Args:
        url: URL to download from
        destination: Path where to save the file
        description: Description for progress bar
    
    Returns:
        Path to downloaded file
    """
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        destination.parent.mkdir(parents=True, exist_ok=True)
        
        with open(destination, 'wb') as file:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=description) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
                        pbar.update(len(chunk))
        
        print(f"✓ Downloaded: {destination.name}")
        return destination
    except Exception as e:
        print(f"✗ Error downloading {url}: {e}")
        if destination.exists():
            destination.unlink()
        return None


def extract_archive(archive_path: Path, extract_to: Path) -> Path:
    """
    Extract a zip or tar archive.
    
    Args:
        archive_path: Path to the archive file
        extract_to: Directory where to extract
    
    Returns:
        Path to extraction directory
    """
    extract_to.mkdir(parents=True, exist_ok=True)
    
    print(f"Extracting {archive_path.name}...")
    
    if archive_path.suffix == '.zip':
        with zipfile.ZipFile(archive_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    elif archive_path.suffix in ['.tar', '.gz', '.tgz']:
        with tarfile.open(archive_path, 'r:*') as tar_ref:
            tar_ref.extractall(extract_to)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path.suffix}")
    
    print(f"✓ Extracted to: {extract_to}")
    return extract_to


def count_video_files(directory: Path) -> dict:
    """
    Count video files in a directory.
    
    Args:
        directory: Directory to search
    
    Returns:
        Dictionary with counts by extension
    """
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.mpg']
    counts = {}
    total = 0
    
    for ext in video_extensions:
        files = list(directory.rglob(f'*{ext}'))
        if files:
            counts[ext] = len(files)
            total += len(files)
    
    counts['total'] = total
    return counts


def generate_ur_fall_urls() -> Dict[str, List[str]]:
    """
    Generate download URLs for UR Fall Detection dataset.
    
    Returns:
        Dictionary with 'fall' and 'adl' keys containing lists of video URLs
    """
    base_url = "https://fenix.ur.edu.pl/mkepski/ds/data/"
    
    urls = {
        'fall': [],
        'adl': []
    }
    
    # Fall sequences: 30 sequences, 2 cameras each (cam0, cam1)
    for i in range(1, 31):
        for cam in [0, 1]:
            video_name = f"fall-{i:02d}-cam{cam}.mp4"
            urls['fall'].append(base_url + video_name)
    
    # ADL sequences: 40 sequences, 1 camera only (cam0)
    for i in range(1, 41):
        video_name = f"adl-{i:02d}-cam0.mp4"
        urls['adl'].append(base_url + video_name)
    
    return urls

## 4. Download UR Fall Detection Dataset

**Dataset Information:**
- 70 sequences total
- 30 fall sequences
- 40 ADL (Activities of Daily Living) sequences
- RGB + Depth data available
- Resolution: 640×480 @ 30fps

In [ ]:
# UR Fall Dataset configuration
ur_fall_base_url = "https://fenix.ur.edu.pl/mkepski/ds/data/"
ur_fall_destination = paths['data_raw'] / 'ur_fall'

print("=" * 60)
print("UR Fall Detection Dataset Download")
print("=" * 60)
print(f"\nBase URL: {ur_fall_base_url}")
print(f"Destination: {ur_fall_destination}")
print("\nDataset structure:")
print("  - 30 fall sequences (60 videos: 2 cameras per sequence)")
print("  - 40 ADL sequences (40 videos: 1 camera per sequence)")
print("  - Total: 100 video files")
print()

In [ ]:
# Generate download URLs
print("Generating download URLs...")
ur_fall_urls = generate_ur_fall_urls()

print(f"\n✓ Generated {len(ur_fall_urls['fall'])} fall video URLs")
print(f"✓ Generated {len(ur_fall_urls['adl'])} ADL video URLs")
print(f"  Total: {len(ur_fall_urls['fall']) + len(ur_fall_urls['adl'])} videos")

# Preview URLs
print("\nSample URLs:")
print(f"  Fall: {ur_fall_urls['fall'][0]}")
print(f"  ADL:  {ur_fall_urls['adl'][0]}")

In [ ]:
# Execute download
# Set to True to start download, False to skip
START_DOWNLOAD = False

if START_DOWNLOAD:
    if IN_COLAB:
        # Download to temporary location, then move to Drive
        download_stats = download_ur_fall_dataset(
            ur_fall_urls, 
            destination=paths['data_raw'],
            download_temp=download_dir
        )
    else:
        # Download directly to destination
        download_stats = download_ur_fall_dataset(
            ur_fall_urls,
            destination=paths['data_raw']
        )
    
    # Print summary
    print("\n" + "=" * 60)
    print("DOWNLOAD COMPLETE")
    print("=" * 60)
    print(f"\nFall videos downloaded: {download_stats['fall_downloaded']}")
    print(f"Fall videos failed: {download_stats['fall_failed']}")
    print(f"ADL videos downloaded: {download_stats['adl_downloaded']}")
    print(f"ADL videos failed: {download_stats['adl_failed']}")
    
    total_downloaded = download_stats['fall_downloaded'] + download_stats['adl_downloaded']
    total_failed = download_stats['fall_failed'] + download_stats['adl_failed']
    
    print(f"\nTotal downloaded: {total_downloaded}")
    print(f"Total failed: {total_failed}")
    
    if total_failed == 0:
        print("\n✓ All videos downloaded successfully!")
    else:
        print(f"\n⚠ {total_failed} videos failed to download. You may need to retry.")
else:
    print("\n⚠ Download skipped. Set START_DOWNLOAD = True to begin download.")

In [ ]:
# Download UR Fall dataset
def download_ur_fall_dataset(urls: Dict[str, List[str]], destination: Path, 
                             download_temp: Path = None) -> Dict[str, int]:
    """
    Download UR Fall Detection dataset videos.
    
    Args:
        urls: Dictionary with 'fall' and 'adl' video URLs
        destination: Final destination directory (e.g., Google Drive)
        download_temp: Temporary download location (for Colab)
    
    Returns:
        Dictionary with download statistics
    """
    if download_temp is None:
        download_temp = destination
    
    stats = {
        'fall_downloaded': 0,
        'adl_downloaded': 0,
        'fall_failed': 0,
        'adl_failed': 0,
    }
    
    # Create directories
    fall_temp_dir = download_temp / 'ur_fall' / 'fall'
    adl_temp_dir = download_temp / 'ur_fall' / 'adl'
    fall_temp_dir.mkdir(parents=True, exist_ok=True)
    adl_temp_dir.mkdir(parents=True, exist_ok=True)
    
    # Download fall sequences
    print("\n" + "=" * 60)
    print("Downloading Fall Sequences")
    print("=" * 60)
    for i, url in enumerate(urls['fall'], 1):
        filename = url.split('/')[-1]
        dest_path = fall_temp_dir / filename
        
        # Skip if already exists
        if dest_path.exists():
            print(f"[{i}/{len(urls['fall'])}] Skipping {filename} (already exists)")
            stats['fall_downloaded'] += 1
            continue
        
        print(f"\n[{i}/{len(urls['fall'])}] Downloading {filename}...")
        result = download_file(url, dest_path, description=filename)
        
        if result:
            stats['fall_downloaded'] += 1
        else:
            stats['fall_failed'] += 1
        
        # Small delay to avoid overwhelming the server
        time.sleep(0.5)
    
    # Download ADL sequences
    print("\n" + "=" * 60)
    print("Downloading ADL Sequences")
    print("=" * 60)
    for i, url in enumerate(urls['adl'], 1):
        filename = url.split('/')[-1]
        dest_path = adl_temp_dir / filename
        
        # Skip if already exists
        if dest_path.exists():
            print(f"[{i}/{len(urls['adl'])}] Skipping {filename} (already exists)")
            stats['adl_downloaded'] += 1
            continue
        
        print(f"\n[{i}/{len(urls['adl'])}] Downloading {filename}...")
        result = download_file(url, dest_path, description=filename)
        
        if result:
            stats['adl_downloaded'] += 1
        else:
            stats['adl_failed'] += 1
        
        # Small delay to avoid overwhelming the server
        time.sleep(0.5)
    
    # Move to final destination if using temporary location
    if download_temp != destination:
        print("\n" + "=" * 60)
        print("Moving files to Google Drive...")
        print("=" * 60)
        
        final_fall_dir = destination / 'ur_fall' / 'fall'
        final_adl_dir = destination / 'ur_fall' / 'adl'
        final_fall_dir.mkdir(parents=True, exist_ok=True)
        final_adl_dir.mkdir(parents=True, exist_ok=True)
        
        # Move fall videos
        for video in fall_temp_dir.glob('*.mp4'):
            shutil.move(str(video), str(final_fall_dir / video.name))
            print(f"✓ Moved {video.name} to Drive")
        
        # Move ADL videos
        for video in adl_temp_dir.glob('*.mp4'):
            shutil.move(str(video), str(final_adl_dir / video.name))
            print(f"✓ Moved {video.name} to Drive")
        
        print("\n✓ All files moved to Google Drive")
    
    return stats


# Start download
print("\n" + "=" * 60)
print("Starting UR Fall Dataset Download")
print("=" * 60)
print("\nThis will download ~100 video files.")
print("Estimated time: 15-30 minutes depending on connection speed.")
print("\nNote: Downloads will resume if interrupted.")

# Confirm before starting
print("\n⚠ Ready to start download? This will take some time.")
print("Files will be downloaded to:", download_dir if IN_COLAB else ur_fall_destination)

## 5. Download Le2i Fall Detection Dataset

**Dataset Information:**
- ~200 videos
- Multiple camera angles and environments
- Includes falls and ADL activities
- Various resolutions and frame rates

In [ ]:
# Le2i Dataset URLs
# Visit: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html

le2i_url = "https://imvia.u-bourgogne.fr/database/FallDataset.zip"  # Example URL - verify on website
le2i_destination = paths['data_raw'] / 'le2i'

print("=" * 60)
print("Le2i Fall Detection Dataset Download")
print("=" * 60)
print("\nDataset source: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html")
print("\nThis dataset may also require registration or manual download.")
if IN_COLAB:
    print(f"Upload to: MyDrive/{GOOGLE_DRIVE_PROJECT_FOLDER}/data/raw/le2i/")
else:
    print(f"Or place in: {le2i_destination}")
print()

In [ ]:
# TODO: Implement Le2i dataset download
# This section will be completed based on the actual download method

print("⚠ Le2i dataset download needs to be implemented")
print("Please download manually for now.")

## 6. Organize Dataset Structure

Ensure datasets are organized in the expected structure:

```
data/raw/
├── ur_fall/
│   ├── fall/
│   │   ├── video_001.avi
│   │   └── ...
│   └── adl/
│       ├── video_001.avi
│       └── ...
└── le2i/
    ├── fall/
    │   └── ...
    └── adl/
        └── ...
```

In [ ]:
def organize_dataset(dataset_path: Path, dataset_name: str):
    """
    Organize dataset into fall/adl subdirectories if not already organized.
    
    Args:
        dataset_path: Path to dataset directory
        dataset_name: Name of the dataset (for logging)
    """
    fall_dir = dataset_path / 'fall'
    adl_dir = dataset_path / 'adl'
    
    fall_dir.mkdir(parents=True, exist_ok=True)
    adl_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nOrganizing {dataset_name} dataset...")
    print(f"  Fall directory: {fall_dir}")
    print(f"  ADL directory: {adl_dir}")
    
    # TODO: Implement organization logic based on actual dataset structure
    # This will depend on how the datasets are originally organized
    
    print(f"✓ {dataset_name} dataset organized")


# Organize both datasets
# organize_dataset(paths['data_raw'] / 'ur_fall', 'UR Fall')
# organize_dataset(paths['data_raw'] / 'le2i', 'Le2i')

print("⚠ Dataset organization logic needs to be implemented")

## 7. Verify Downloads and Create Metadata

In [ ]:
def verify_dataset(dataset_path: Path, dataset_name: str) -> dict:
    """
    Verify dataset download and count files.
    
    Args:
        dataset_path: Path to dataset directory
        dataset_name: Name of the dataset
    
    Returns:
        Dictionary with verification results
    """
    print(f"\nVerifying {dataset_name} dataset...")
    print(f"Location: {dataset_path}")
    
    if not dataset_path.exists():
        print(f"✗ Directory does not exist: {dataset_path}")
        return {'exists': False}
    
    fall_dir = dataset_path / 'fall'
    adl_dir = dataset_path / 'adl'
    
    fall_count = count_video_files(fall_dir) if fall_dir.exists() else {'total': 0}
    adl_count = count_video_files(adl_dir) if adl_dir.exists() else {'total': 0}
    
    results = {
        'exists': True,
        'dataset': dataset_name,
        'path': str(dataset_path),
        'fall_videos': fall_count['total'],
        'adl_videos': adl_count['total'],
        'total_videos': fall_count['total'] + adl_count['total'],
    }
    
    print(f"  Fall videos: {results['fall_videos']}")
    print(f"  ADL videos: {results['adl_videos']}")
    print(f"  Total videos: {results['total_videos']}")
    
    if results['total_videos'] > 0:
        print(f"✓ {dataset_name} dataset verified")
    else:
        print(f"⚠ No videos found in {dataset_name} dataset")
    
    return results

In [ ]:
# Verify all datasets
print("=" * 60)
print("Dataset Verification")
print("=" * 60)

ur_fall_info = verify_dataset(paths['data_raw'] / 'ur_fall', 'UR Fall')
le2i_info = verify_dataset(paths['data_raw'] / 'le2i', 'Le2i')

In [ ]:
# Create summary metadata
metadata = [
    ur_fall_info,
    le2i_info
]

df_metadata = pd.DataFrame(metadata)
print("\n" + "=" * 60)
print("Dataset Summary")
print("=" * 60)
print(df_metadata.to_string(index=False))

# Save metadata
metadata_path = paths['data_raw'] / 'dataset_metadata.csv'
df_metadata.to_csv(metadata_path, index=False)
print(f"\n✓ Metadata saved to: {metadata_path}")

## 8. Summary and Next Steps

In [ ]:
total_fall_videos = ur_fall_info.get('fall_videos', 0) + le2i_info.get('fall_videos', 0)
total_adl_videos = ur_fall_info.get('adl_videos', 0) + le2i_info.get('adl_videos', 0)
total_videos = total_fall_videos + total_adl_videos

print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)
print(f"\nTotal fall videos: {total_fall_videos}")
print(f"Total ADL videos: {total_adl_videos}")
print(f"Total videos: {total_videos}")

if total_videos > 0:
    print(f"\nClass balance: {total_fall_videos / total_videos * 100:.1f}% falls, {total_adl_videos / total_videos * 100:.1f}% ADL")

if IN_COLAB:
    print(f"\n✓ Datasets saved in Google Drive at: {paths['data_raw']}")
    print(f"  (MyDrive/{GOOGLE_DRIVE_PROJECT_FOLDER}/data/raw/)")
else:
    print(f"\n✓ Datasets saved locally at: {paths['data_raw']}")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("1. Review downloaded data in 01_eda.ipynb")
print("2. Extract pose keypoints using 02_pose_extraction.ipynb")
print("3. Train models using 03_model_experiments.ipynb")
print("\nNote: If datasets need to be downloaded manually:")
print("  - UR Fall: http://fenix.ur.edu.pl/~mkepski/ds/uf.html")
print("  - Le2i: https://imvia.u-bourgogne.fr/en/database/fall-detection-dataset-2.html")